# Find burial mounds in Romania's LiDAR

**Press ▶ on the cell below** (or Runtime → Run all). In about a minute a map appears under it.

**Find mounds:** search a village (top right of the map) or click inside the **green area**, then press **▶ Scan**.
You get the relief of the square and the places that look like burial mounds:
🔴 red = probable mound · 🟠 orange = worth a second look.

**Test yourself:** choose *Test yourself* (top left of the map) and scan. Click every mound you can see on the relief,
then press *Show the model's answer*: you see what you both found, what only you found and what only the model found.

Shape is not proof: only a field visit confirms a mound. Please do not dig; report finds to the county heritage office.

> ANCPI's LiDAR server is offline for now; the whole 0.5 m coverage (Dolj, Gorj, Mehedinți, Caraș-Severin) works from this project's copy of the tiles (© ANCPI, unmodified).

How it works and how well: [STUDY.md](https://github.com/ObuObuHub/tumulus-lidar-detector/blob/main/STUDY.md).

In [ ]:
#@title ▶ Press play — the map appears below in about a minute { display-mode: "form" }
# Everything runs in this one cell: it downloads the project, then shows the map.
import os, sys, json, base64, subprocess, html, traceback
BRANCH = 'colab-v2'
REPO = 'https://github.com/ObuObuHub/tumulus-lidar-detector.git'

def _sh(cmd, what):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'{what} failed (exit {r.returncode}):\n{(r.stdout + r.stderr)[-2000:]}')

print('Downloading the project…', flush=True)
if os.path.basename(os.getcwd()) != 'tumulus-lidar-detector':
    if not os.path.isdir('tumulus-lidar-detector'):
        _sh(['git', 'clone', '-q', '--depth', '1', '-b', BRANCH, REPO], 'Downloading the project')
    os.chdir('tumulus-lidar-detector')
# a reused session gets the current version, not a stale copy
_sh(['git', 'fetch', '-q', '--depth', '1', 'origin', BRANCH], 'Updating the project')
_sh(['git', 'checkout', '-q', '-f', '-B', BRANCH, 'FETCH_HEAD'], 'Updating the project')
try:
    import pyproj
except ImportError:
    _sh([sys.executable, '-m', 'pip', 'install', '-q', 'pyproj'], 'Installing pyproj')
    import pyproj

import IPython
from IPython.display import display, HTML
from google.colab import output as _colab_output
_tf_to3844 = pyproj.Transformer.from_crs('EPSG:4326', 'EPSG:3844', always_xy=True)
_tf_to4326 = pyproj.Transformer.from_crs('EPSG:3844', 'EPSG:4326', always_xy=True)
_cov_rects = json.load(open('assets/laki3_coverage.geojson'))            # exact, for the point check
_cov_outline = json.load(open('assets/laki3_coverage_outline.geojson'))  # dissolved, for display
_mir_outline = json.load(open('assets/mirror_coverage_outline.geojson'))  # tile mirror: works while ANCPI is offline

def _inside(lon, lat):
    for f in _cov_rects['features']:
        ring = f['geometry']['coordinates'][0]
        c = False; n = len(ring); j = n - 1
        for i in range(n):
            xi, yi = ring[i]; xj, yj = ring[j]
            if ((yi > lat) != (yj > lat)) and (lon < (xj - xi) * (lat - yi) / (yj - yi) + xi):
                c = not c
            j = i
        if c:
            return True
    return False

def _fail(msg, detail=''):
    out = {'ok': False, 'msg': msg}
    if detail:
        out['detail'] = html.escape(detail[-2500:])
    return IPython.display.JSON(out)

def _b64(path):
    return base64.b64encode(open(path, 'rb').read()).decode()

def _scan_inner(lat, lon, km, test):
    lat = float(lat); lon = float(lon); km = max(1, min(3, int(km)))
    if not _inside(lon, lat):
        return _fail('This point is outside the LiDAR coverage (the blue area). Pick a point inside the green area.')
    for _f in ('/tmp/zone_dets.csv', '/tmp/zone_cover.json', '/tmp/zone_px.json', 'review/zone_view.jpg',
               'review/zone_relief.jpg', 'review/zone_board.jpg', 'detected_mounds.csv'):
        if os.path.exists(_f):
            os.remove(_f)
    try:
        r = subprocess.run([sys.executable, 'tools/scan_zone_v4.py', str(lon), str(lat), str(km)],
                           capture_output=True, text=True, timeout=1200)
    except subprocess.TimeoutExpired:
        return _fail('The scan took longer than 20 minutes and was stopped. Try a 1 km square.')
    log = (r.stdout or '') + (r.stderr or '')
    if r.returncode == 2 and 'no LiDAR tiles here' in log:
        return _fail('No LiDAR tiles for this square right now. ANCPI is offline; the green area works '
                     '(Dolj, Gorj, Mehedinți, Caraș-Severin).')
    if r.returncode != 0:
        return _fail(f'The scanner stopped with an error (exit code {r.returncode}). Its last lines are below.', log)
    for need in ('review/zone_view.jpg', 'review/zone_relief.jpg', '/tmp/zone_dets.csv', '/tmp/zone_px.json'):
        if not os.path.exists(need):
            return _fail(f'The scanner finished but did not write {need}. Its last lines are below.', log)
    import pandas as pd
    det = pd.read_csv('/tmp/zone_dets.csv')
    rows = []; maybe = []
    kept = det[det['keep'] == 1].sort_values('score', ascending=False)
    for i, (_, x) in enumerate(kept.iterrows(), 1):
        rows.append({'id': i, 'lon': round(float(x.lon), 6), 'lat': round(float(x.lat), 6), 'score': round(float(x.score), 3)})
    brd = det[(det['keep'] == 0) & (det['score'] >= 0.60)].sort_values('score', ascending=False).head(15)
    for _, x in brd.iterrows():
        maybe.append({'lon': round(float(x.lon), 6), 'lat': round(float(x.lat), 6), 'score': round(float(x.score), 3)})
    est, nord = _tf_to3844.transform(lon, lat); _half = km * 500
    _c1 = _tf_to4326.transform(est - _half, nord - _half); _c2 = _tf_to4326.transform(est + _half, nord + _half)
    bbox = [round(min(_c1[1], _c2[1]), 6), round(min(_c1[0], _c2[0]), 6),
            round(max(_c1[1], _c2[1]), 6), round(max(_c1[0], _c2[0]), 6)]
    cover = json.load(open('/tmp/zone_cover.json')) if os.path.exists('/tmp/zone_cover.json') else None
    px = json.load(open('/tmp/zone_px.json'))
    csv64 = _b64('detected_mounds.csv') if os.path.exists('detected_mounds.csv') else ''
    img = _b64('review/zone_relief.jpg' if test else 'review/zone_view.jpg')
    return IPython.display.JSON({'ok': True, 'n': len(rows), 'img': img, 'rows': rows, 'maybe': maybe, 'bbox': bbox,
                                 'cover': cover, 'csv': csv64, 'px': px, 'test': bool(test)})

def _scan(lat, lon, km=1, test=False):
    try:
        return _scan_inner(lat, lon, km, test)
    except Exception as e:
        return _fail(f'The scan stopped with an error: {type(e).__name__}: {e}', traceback.format_exc())

_colab_output.register_callback('notebook.scan', _scan)
print('✔ Ready.', flush=True)
display(HTML(open('assets/demo_map.html').read().replace('__COVOUT__', json.dumps(_cov_outline)).replace('__MIROUT__', json.dumps(_mir_outline))))